# ASG Airlines — End-to-End Data Engineering Pipeline

## Objective
Build an end-to-end data engineering pipeline for ASG Airlines to:
- Ingest raw airline data
- Perform data quality checks
- Clean and standardize the data
- Handle missing and duplicate records
- Handle overnight flights
- Calculate flight duration
- Protect passenger PII
- Identify anomalies
- Generate analytical datasets
- Calculate business KPIs
- Prepare data for Power BI

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
file=pd.ExcelFile(r"C:\Users\ajint\Downloads\Airlines Use Case (1)\Airlines Use Case\UseCase - Airlines.xlsx")
print(file.sheet_names)

['flights', 'payments', 'bookings', 'passengers']


In [4]:
flights=pd.read_excel(file,sheet_name="flights")
payments=pd.read_excel(file,sheet_name="payments")
bookings=pd.read_excel(file,sheet_name="bookings")
passengers=pd.read_excel(file,sheet_name="passengers")

In [5]:
print("Flights:",flights.shape)
print("Bookings:",bookings.shape)
print("Payments:",payments.shape)
print("Passengers:",passengers.shape)

Flights: (1020, 7)
Bookings: (1000, 9)
Payments: (1000, 4)
Passengers: (1039, 9)


In [6]:
print("FLIGHTS",flights.columns.tolist())
print("BOOKINGS",bookings.columns.tolist())
print("PAYMENTS",payments.columns.tolist())
print("PASSENGERS",passengers.columns.tolist())

FLIGHTS ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']
BOOKINGS ['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'passport_number', 'seat_number', 'emergency_contact_name', 'emergency_contact_phone']
PAYMENTS ['payment_id', 'booking_id', 'amount', 'payment_method']
PASSENGERS ['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'email', 'phone', 'aadhaar_id', 'date_of_birth']


In [7]:
print("FLIGHTS:")
print(flights.info())
print("BOOKINGS:")
print(bookings.info())
print("PAYMENTS:")
print(payments.info())
print("PASSENGERS:")
print(passengers.info())

FLIGHTS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   flight_id       1020 non-null   object        
 1   airline         979 non-null    object        
 2   source          1020 non-null   object        
 3   destination     1020 non-null   object        
 4   departure_time  1020 non-null   datetime64[ns]
 5   arrival_time    1020 non-null   datetime64[ns]
 6   duration        1020 non-null   object        
dtypes: datetime64[ns](2), object(5)
memory usage: 55.9+ KB
None
BOOKINGS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   booking_id               1000 non-null   object        
 1   passenger_id             1000 non-null   object  

In [8]:
print("Flight  missing values")
print(flights.isnull().sum())
print("\nBookings Missing values")
print(bookings.isnull().sum())
print("\nPayments missing values")
print(payments.isnull().sum())
print("\nPassengers missing values")
print(passengers.isnull().sum())

Flight  missing values
flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

Bookings Missing values
booking_id                  0
passenger_id                0
flight_id                   0
booking_date                0
status                     45
passport_number             0
seat_number                 0
emergency_contact_name      0
emergency_contact_phone     0
dtype: int64

Payments missing values
payment_id         0
booking_id         0
amount            48
payment_method     0
dtype: int64

Passengers missing values
passenger_id      0
first_name        0
last_name        10
age               0
gender            0
email             0
phone             0
aadhaar_id        0
date_of_birth     0
dtype: int64


In [9]:
print("Flights duplicate rows:", flights.duplicated().sum())
print("Bookings duplicate rows:", bookings.duplicated().sum())
print("Payments duplicate rows:", payments.duplicated().sum())
print("Passengers duplicate rows:", passengers.duplicated().sum())

Flights duplicate rows: 15
Bookings duplicate rows: 0
Payments duplicate rows: 0
Passengers duplicate rows: 0


In [10]:
duplicate_flights=flights[flights.duplicated(keep=False)].sort_values("flight_id")
duplicate_flights

,flight_id,airline,source,destination,departure_time,arrival_time,duration
550,AI020,NaN,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,02:28:00
549,AI020,NaN,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,02:28:00
141,AI031,Air India,DEL,MAA,2026-04-20 13:05:41.701,2026-04-20 16:11:41.701,03:06:00
142,AI031,Air India,DEL,MAA,2026-04-20 13:05:41.701,2026-04-20 16:11:41.701,03:06:00
598,AI043,Air India,CCU,DEL,2026-04-19 00:28:41.701,2026-04-19 04:37:41.701,04:09:00
599,AI043,Air India,CCU,DEL,2026-04-19 00:28:41.701,2026-04-19 04:37:41.701,04:09:00
603,AI070,Air India,CCU,DEL,2026-04-19 00:05:41.702,2026-04-19 01:44:41.702,01:39:00
602,AI070,Air India,CCU,DEL,2026-04-19 00:05:41.702,2026-04-19 01:44:41.702,01:39:00
83,AI242,UNKNOWN,BLR,CCU,2026-04-20 16:41:41.704,2026-04-20 17:43:41.704,01:02:00
84,AI242,UNKNOWN,BLR,CCU,2026-04-20 16:41:41.704,2026-04-20 17:43:41.704,01:02:00


In [11]:
print("Number of duplicate flight rows:",len(duplicate_flights))

Number of duplicate flight rows: 30


In [12]:
flight_id_counts = flights["flight_id"].value_counts()
flight_id_counts[flight_id_counts>1]

flight_id
UK049    2
SJ118    2
SJ142    2
UK180    2
UK013    2
AI070    2
AI043    2
UK160    2
SJ146    2
AI242    2
SJ037    2
AI020    2
UK139    2
UK163    2
AI031    2
6F250    2
Name: count, dtype: int64

In [13]:
duplicate_ids=flight_id_counts[flight_id_counts>1].index
flights[flights["flight_id"].isin(duplicate_ids)].sort_values("flight_id")

,flight_id,airline,source,destination,departure_time,arrival_time,duration
253,6F250,UNKNOWN,DEL,BLR,2026-04-20 03:26:41.701,2026-04-20 07:30:41.701,04:04:00
270,6F250,UNKNOWN,CCU,BLR,2026-04-20 02:23:41.702,2026-04-20 02:56:41.702,00:33:00
550,AI020,NaN,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,02:28:00
549,AI020,NaN,BOM,BLR,2026-04-19 03:53:41.701,2026-04-19 06:21:41.701,02:28:00
141,AI031,Air India,DEL,MAA,2026-04-20 13:05:41.701,2026-04-20 16:11:41.701,03:06:00
142,AI031,Air India,DEL,MAA,2026-04-20 13:05:41.701,2026-04-20 16:11:41.701,03:06:00
599,AI043,Air India,CCU,DEL,2026-04-19 00:28:41.701,2026-04-19 04:37:41.701,04:09:00
598,AI043,Air India,CCU,DEL,2026-04-19 00:28:41.701,2026-04-19 04:37:41.701,04:09:00
603,AI070,Air India,CCU,DEL,2026-04-19 00:05:41.702,2026-04-19 01:44:41.702,01:39:00
602,AI070,Air India,CCU,DEL,2026-04-19 00:05:41.702,2026-04-19 01:44:41.702,01:39:00


In [14]:
print(flights["flight_id"].astype(str).sort_values().to_string(index=False))

6F001
6F002
6F003
6F004
6F005
6F006
6F007
6F008
6F009
6F010
6F011
6F012
6F013
6F014
6F015
6F016
6F017
6F018
6F019
6F020
6F021
6F022
6F023
6F024
6F025
6F026
6F027
6F028
6F029
6F030
6F031
6F032
6F033
6F034
6F035
6F036
6F037
6F038
6F039
6F040
6F041
6F042
6F043
6F044
6F045
6F046
6F047
6F048
6F049
6F050
6F051
6F052
6F053
6F054
6F055
6F056
6F057
6F058
6F059
6F060
6F061
6F062
6F063
6F064
6F065
6F066
6F067
6F068
6F069
6F070
6F071
6F072
6F073
6F074
6F075
6F076
6F077
6F078
6F079
6F080
6F081
6F082
6F083
6F084
6F085
6F086
6F087
6F088
6F089
6F090
6F091
6F092
6F093
6F094
6F095
6F096
6F097
6F098
6F099
6F100
6F101
6F102
6F103
6F104
6F105
6F106
6F107
6F108
6F109
6F110
6F111
6F112
6F113
6F114
6F115
6F116
6F117
6F118
6F119
6F120
6F121
6F122
6F123
6F124
6F125
6F126
6F127
6F128
6F129
6F130
6F131
6F132
6F133
6F134
6F135
6F136
6F137
6F138
6F139
6F140
6F141
6F142
6F143
6F144
6F145
6F146
6F147
6F148
6F149
6F150
6F151
6F152
6F153
6F154
6F155
6F156
6F157
6F158
6F159
6F160
6F161
6F162
6F163
6F164
6F165
6F166
6F16

In [15]:
booking_flight_valid = bookings["flight_id"].astype("string").str.match(r"^(?:[A-Z]{2}|\d[A-Z])\d{3}$",na=False)
print("Invalid booking flight IDs:",(~booking_flight_valid).sum())

Invalid booking flight IDs: 0


In [16]:
print(flights["airline"].value_counts(dropna=False))

airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           41
UNKNOWN       31
Name: count, dtype: int64


In [17]:
# flights["airline"] = flights["airline"].astype("string").str.strip()

In [18]:
print(flights["airline"].value_counts(dropna=False))

airline
IndiGo       249
SpiceJet     240
Air India    236
Vistara      223
NaN           41
UNKNOWN       31
Name: count, dtype: int64


In [19]:
# 1. Airline code mapping
airline_code_map = {
    "UK": "Vistara",
    "AI": "Air India",
    "6F": "IndiGo",
    "SJ": "SpiceJet",
}

# 2. Clean the airline column
flights["airline"] = (
    flights["airline"]
    .astype("string")
    .str.strip()
)

# 3. Treat NaN and Unknown as missing
flights["airline"] = flights["airline"].replace(
    ["Unknown", "UNKNOWN", "unknown", ""],
    pd.NA
)

# 4. Extract the airline code from flight_id
flights["flight_code"] = (
    flights["flight_id"]
    .astype("string")
    .str.strip()
    .str[:2]
    .str.upper()
)

# 5. Identify rows where airline is missing
missing_airline = flights["airline"].isna()

# 6. Fill missing airline using flight code
flights.loc[missing_airline, "airline"] = (
    flights.loc[missing_airline, "flight_code"]
    .map(airline_code_map)
)

In [20]:
print(
    flights[
        flights["airline"].isna()
    ][["flight_id", "flight_code", "airline"]]
)

Empty DataFrame
Columns: [flight_id, flight_code, airline]
Index: []


In [21]:
flights["source"] = (
    flights["source"]
    .astype("string")
    .str.strip()
    .str.upper()
)

flights["destination"] = (
    flights["destination"]
    .astype("string")
    .str.strip()
    .str.upper())

In [22]:
flights["source"] = (
    flights["source"]
    .astype("string")
    .str.strip()
    .str.upper()
)

flights["destination"] = (
    flights["destination"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [23]:
print(flights["source"].value_counts(dropna=False))
print()
print(flights["destination"].value_counts(dropna=False))

source
BOM    207
HYD    180
CCU    174
DEL    163
MAA    157
BLR    139
Name: count, dtype: Int64

destination
DEL    200
CCU    188
BOM    174
BLR    165
MAA    151
HYD    142
Name: count, dtype: Int64


In [24]:
flights["route"] = (flights["source"] + " → " + flights["destination"])

In [25]:
flights[[
    "flight_id",
    "source",
    "destination",
    "route"
]].head()

,flight_id,source,destination,route
0,SJ010,CCU,MAA,CCU → MAA
1,AI155,BOM,CCU,BOM → CCU
2,UK094,BOM,CCU,BOM → CCU
3,AI245,BOM,CCU,BOM → CCU
4,AI192,MAA,BOM,MAA → BOM


In [26]:
flights["departure_time"] = pd.to_datetime(
    flights["departure_time"],
    format="%d-%m-%Y %H:%M:%S",
    errors="coerce")

flights["arrival_time"] = pd.to_datetime(
    flights["arrival_time"],
    format="%d-%m-%Y %H:%M:%S",
    errors="coerce"
)

In [29]:
print(flights[[
    "departure_time",
    "arrival_time"
]].dtypes)

departure_time    datetime64[ns]
arrival_time      datetime64[ns]
dtype: object


In [30]:
print("Missing departure times:",
      flights["departure_time"].isna().sum())

print("Missing arrival times:",
      flights["arrival_time"].isna().sum())

Missing departure times: 0
Missing arrival times: 0


In [31]:
flights["calculated_duration"] = (
    flights["arrival_time"] -
    flights["departure_time"]
)

In [32]:
flights[[
    "flight_id",
    "departure_time",
    "arrival_time",
    "calculated_duration"
]].head(10)

,flight_id,departure_time,arrival_time,calculated_duration
0,SJ010,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,0 days 02:54:00
1,AI155,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,0 days 01:48:00
2,UK094,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,0 days 01:45:00
3,AI245,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,0 days 02:36:00
4,AI192,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,0 days 04:59:00
5,SJ158,2026-04-20 23:05:41.703,2026-04-21 01:24:41.703,0 days 02:19:00
6,6F196,2026-04-20 23:04:41.703,2026-04-21 00:47:41.703,0 days 01:43:00
7,AI080,2026-04-20 23:03:41.702,2026-04-21 00:35:41.702,0 days 01:32:00
8,6F025,2026-04-20 23:02:41.701,2026-04-20 23:57:41.701,0 days 00:55:00
9,6F251,2026-04-20 22:56:41.703,2026-04-20 23:37:41.703,0 days 00:41:00


In [33]:
flights["duration_minutes"] = (
    flights["calculated_duration"]
    .dt.total_seconds() / 60
)

In [34]:
flights[[
    "flight_id",
    "duration_minutes"
]].head()

,flight_id,duration_minutes
0,SJ010,174.0
1,AI155,108.0
2,UK094,105.0
3,AI245,156.0
4,AI192,299.0


In [35]:
print(flights["duration_minutes"].describe())

count    1020.000000
mean      162.919739
std        87.335793
min     -1140.000000
25%        99.000000
50%       165.000000
75%       232.000000
max       300.000000
Name: duration_minutes, dtype: float64


In [36]:
flights[
    (flights["duration_minutes"] <= 0) |
    (flights["duration_minutes"] > 24 * 60)
][[
    "flight_id",
    "source",
    "destination",
    "departure_time",
    "arrival_time",
    "duration_minutes"
]]

,flight_id,source,destination,departure_time,arrival_time,duration_minutes
355,SJ192,HYD,BOM,2026-04-19 18:45:42,2026-04-18 23:45:42,-1140.0


In [37]:
import datetime
import numpy as np

# 1. Ensure departure and arrival are converted to datetime
flights['departure_time'] = pd.to_datetime(flights['departure_time'], errors='coerce')
flights['arrival_time'] = pd.to_datetime(flights['arrival_time'], errors='coerce')

# 2. Calculate duration from timestamps (handling overnight/cross-day flights)
def calculate_duration(row):
    dep = row['departure_time']
    arr = row['arrival_time']
    if pd.isnull(dep) or pd.isnull(arr):
        return np.nan
    # If arrival is earlier than departure, it's an overnight flight
    if arr < dep:
        arr += pd.Timedelta(days=1)
    return round((arr - dep).total_seconds() / 60, 2)

flights['calculated_duration_mins'] = flights.apply(calculate_duration, axis=1)

# 3. Robust parser for the original 'duration' column
def parse_original_duration(val):
    if pd.isnull(val):
        return np.nan
    if isinstance(val, datetime.time):
        return val.hour * 60 + val.minute + val.second / 60
    if isinstance(val, datetime.timedelta):
        return val.total_seconds() / 60
    if isinstance(val, (int, float)):
        return float(val)
    
    val_str = str(val).strip()
    try:
        parts = val_str.split(':')
        if len(parts) == 3:
            return int(parts[0]) * 60 + int(parts[1]) + float(parts[2]) / 60
        elif len(parts) == 2:
            return int(parts[0]) * 60 + int(parts[1])
    except Exception:
        pass
        
    return pd.to_numeric(val, errors='coerce')

flights["duration_minutes_original"] = flights["duration"].apply(parse_original_duration)

# 4. Compare both side-by-side cleanly
comparison_df = flights[['flight_id', 'departure_time', 'arrival_time', 'duration_minutes_original', 'calculated_duration_mins']]
print(comparison_df.head())

  flight_id          departure_time            arrival_time  \
0     SJ010 2026-04-20 23:38:41.701 2026-04-21 02:32:41.701   
1     AI155 2026-04-20 23:35:41.703 2026-04-21 01:23:41.703   
2     UK094 2026-04-20 23:26:41.702 2026-04-21 01:11:41.702   
3     AI245 2026-04-20 23:07:41.704 2026-04-21 01:43:41.704   
4     AI192 2026-04-20 23:05:41.703 2026-04-21 04:04:41.703   

   duration_minutes_original  calculated_duration_mins  
0                      174.0                     174.0  
1                      108.0                     108.0  
2                      105.0                     105.0  
3                      156.0                     156.0  
4                      299.0                     299.0  


In [38]:
# Calculate the absolute difference between original and calculated durations
flights["duration_difference"] = flights["duration_minutes_original"] - flights["calculated_duration_mins"]

# Filter rows where the difference is greater than 0.01 minutes (to handle floating-point precision)
duration_errors = flights[
    flights["duration_difference"].abs() > 0.01
]
print("Duration inconsistencies:", len(duration_errors))

Duration inconsistencies: 0


In [39]:
duration_errors[[
    "flight_id",
    "departure_time",
    "arrival_time",
    "duration",
    "duration_minutes_original",
    "duration_minutes"
]]

,flight_id,departure_time,arrival_time,duration,duration_minutes_original,duration_minutes


In [40]:
flights["overnight_flight"] = (
    flights["arrival_time"].dt.date >
    flights["departure_time"].dt.date
)

In [41]:
print(
    flights["overnight_flight"].value_counts()
)

overnight_flight
False    896
True     124
Name: count, dtype: int64


In [42]:
flights["departure_date"] = (
    flights["departure_time"].dt.date
)

In [43]:
flights["departure_hour"] = (
    flights["departure_time"].dt.hour
)

In [44]:
flights["data_quality_status"] = "Valid"

In [45]:
flights["duration_minutes"] = pd.to_numeric(
    flights["duration_minutes"],
    errors="coerce"
)

In [46]:
invalid_duration = (
    flights["duration_minutes"] <= 0
) | (
    flights["duration_minutes"] > 24 * 60
)

flights.loc[
    invalid_duration,
    "data_quality_status"
] = "Invalid Duration"

In [47]:
flights["anomaly_flag"] = (
    flights["data_quality_status"] != "Valid"
)

In [48]:
flights["anomaly_flag"].value_counts()

anomaly_flag
False    1019
True        1
Name: count, dtype: int64

In [49]:
bookings = bookings.copy()

In [50]:
text_columns = [
    "booking_id",
    "passenger_id",
    "flight_id",
    "status",
    "passport_number",
    "seat_number",
    "emergency_contact_name",
    "emergency_contact_phone"
]

for col in text_columns:
    bookings[col] = (
        bookings[col]
        .astype("string")
        .str.strip()
    )

In [51]:
print(
    bookings["status"]
    .value_counts(dropna=False)
)

status
CONFIRMED    320
CANCELLED    314
PENDING      291
<NA>          45
INVALID       30
Name: count, dtype: Int64


In [52]:
bookings["status"] = bookings["status"].fillna("UNKNOWN")

In [53]:
print(bookings["status"].value_counts())

status
CONFIRMED    320
CANCELLED    314
PENDING      291
UNKNOWN       45
INVALID       30
Name: count, dtype: Int64


In [54]:
booking_id_valid = bookings["booking_id"].str.match(
    r"^B\d+$",
    na=False
)

print("Invalid booking IDs:",
      (~booking_id_valid).sum())

Invalid booking IDs: 0


In [55]:
passenger_id_valid = bookings["passenger_id"].str.match(
    r"^P\d+$",
    na=False
)

print(
    "Invalid passenger IDs:",
    (~passenger_id_valid).sum()
)

Invalid passenger IDs: 0


In [56]:
booking_flight_valid = bookings["flight_id"].astype("string").str.match(
    r"^(?:[A-Z]{2}|\d[A-Z])\d{3}$",
    na=False
)

print(
    "Invalid booking flight IDs:",
    (~booking_flight_valid).sum()
)

Invalid booking flight IDs: 0


In [57]:
bookings["booking_date"] = pd.to_datetime(
    bookings["booking_date"],
    errors="coerce"
)

In [58]:
print(bookings["booking_date"].isna().sum())

0


In [59]:
passengers_clean = passengers.copy()

In [60]:
passengers_clean["aadhaar_masked"] = (
    passengers_clean["aadhaar_id"]
    .astype("string")
    .str[-4:]
    .radd("********")
)

In [61]:
passengers_clean["phone_masked"] = (
    passengers_clean["phone"]
    .astype("string")
    .str[-4:]
    .radd("******")
)

In [62]:
passengers_analytics = passengers_clean[
    [
        "passenger_id",
        "age",
        "gender",
        "date_of_birth",
        "aadhaar_masked",
        "phone_masked"
    ]
].copy()

In [63]:
passengers_analytics["gender"] = (
    passengers_analytics["gender"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [64]:
payments = payments.copy()

In [65]:
payments["payment_id"] = (
    payments["payment_id"]
    .astype("string")
    .str.strip()
)

payments["booking_id"] = (
    payments["booking_id"]
    .astype("string")
    .str.strip()
)

In [66]:
print(payments["amount"].dtype)

object


In [67]:
payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

In [68]:
print(payments["amount"].dtype)

float64


In [69]:
print(
    "Missing payment amounts:",
    payments["amount"].isna().sum()
)

Missing payment amounts: 78


In [70]:
print(
    payments["payment_method"]
    .value_counts(dropna=False)
)

payment_method
UPI           358
CARD          329
NETBANKING    313
Name: count, dtype: int64


In [71]:
payments["payment_method"] = (
    payments["payment_method"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [72]:
print(
    payments["payment_method"]
    .value_counts(dropna=False)
)

payment_method
UPI           358
CARD          329
NETBANKING    313
Name: count, dtype: Int64


In [73]:
invalid_payment_bookings = payments[
    ~payments["booking_id"].isin(
        bookings["booking_id"]
    )
]

print(
    "Payments with invalid booking IDs:",
    len(invalid_payment_bookings)
)

Payments with invalid booking IDs: 0


In [74]:
invalid_booking_flights = bookings[
    ~bookings["flight_id"].isin(
        flights["flight_id"]
    )
]

print(
    "Bookings with invalid flight IDs:",
    len(invalid_booking_flights)
)

Bookings with invalid flight IDs: 0


In [75]:
invalid_booking_passengers = bookings[
    ~bookings["passenger_id"].isin(
        passengers["passenger_id"]
    )
]

print(
    "Bookings with invalid passenger IDs:",
    len(invalid_booking_passengers)
)

Bookings with invalid passenger IDs: 0


In [76]:
quality_summary = pd.DataFrame({
    "Dataset": [
        "Flights",
        "Bookings",
        "Payments",
        "Passengers"
    ],
    "Rows": [
        len(flights),
        len(bookings),
        len(payments),
        len(passengers)
    ],
    "Duplicate_Rows": [
        flights.duplicated().sum(),
        bookings.duplicated().sum(),
        payments.duplicated().sum(),
        passengers.duplicated().sum()
    ],
    "Missing_Values": [
        flights.isna().sum().sum(),
        bookings.isna().sum().sum(),
        payments.isna().sum().sum(),
        passengers.isna().sum().sum()
    ]
})

quality_summary

,Dataset,Rows,Duplicate_Rows,Missing_Values
0,Flights,1020,15,2
1,Bookings,1000,0,0
2,Payments,1000,0,78
3,Passengers,1039,0,10


In [77]:
print("===== MISSING VALUES BY COLUMN =====")

print("\nFLIGHTS")
print(flights.isna().sum())

print("\nBOOKINGS")
print(bookings.isna().sum())

print("\nPAYMENTS")
print(payments.isna().sum())

print("\nPASSENGERS")
print(passengers.isna().sum())

===== MISSING VALUES BY COLUMN =====

FLIGHTS
flight_id                    0
airline                      0
source                       0
destination                  0
departure_time               0
arrival_time                 0
duration                     0
flight_code                  0
route                        0
calculated_duration          0
duration_minutes             0
calculated_duration_mins     0
duration_minutes_original    1
duration_difference          1
overnight_flight             0
departure_date               0
departure_hour               0
data_quality_status          0
anomaly_flag                 0
dtype: int64

BOOKINGS
booking_id                 0
passenger_id               0
flight_id                  0
booking_date               0
status                     0
passport_number            0
seat_number                0
emergency_contact_name     0
emergency_contact_phone    0
dtype: int64

PAYMENTS
payment_id         0
booking_id         0
amount          

In [78]:
flights_analytics = flights[
    [
        "flight_id",
        "airline",
        "source",
        "destination",
        "route",
        "departure_time",
        "arrival_time",
        "departure_date",
        "departure_hour",
        "duration_minutes",
        "overnight_flight",
        "anomaly_flag",
        "data_quality_status"
    ]
].copy()

In [79]:
flights_analytics.head()

,flight_id,airline,source,destination,route,departure_time,arrival_time,departure_date,departure_hour,duration_minutes,overnight_flight,anomaly_flag,data_quality_status
0,SJ010,SpiceJet,CCU,MAA,CCU → MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,2026-04-20,23,174.0,True,False,Valid
1,AI155,Air India,BOM,CCU,BOM → CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,2026-04-20,23,108.0,True,False,Valid
2,UK094,Vistara,BOM,CCU,BOM → CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,2026-04-20,23,105.0,True,False,Valid
3,AI245,Air India,BOM,CCU,BOM → CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,2026-04-20,23,156.0,True,False,Valid
4,AI192,Air India,MAA,BOM,MAA → BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,2026-04-20,23,299.0,True,False,Valid


In [80]:
flights_analytics = flights_analytics.drop_duplicates()

In [81]:
print(
    "Remaining duplicate rows:",
    flights_analytics.duplicated().sum()
)

Remaining duplicate rows: 0


In [82]:
bookings_analytics = bookings[
    [
        "booking_id",
        "passenger_id",
        "flight_id",
        "booking_date",
        "status",
        "seat_number"
    ]
].copy()

In [83]:
payments_analytics = payments[
    [
        "payment_id",
        "booking_id",
        "amount",
        "payment_method"
    ]
].copy()

In [84]:
print("Flights Analytics:", flights_analytics.shape)
print("Bookings Analytics:", bookings_analytics.shape)
print("Payments Analytics:", payments_analytics.shape)
print("Passengers Analytics:", passengers_analytics.shape)

Flights Analytics: (1005, 13)
Bookings Analytics: (1000, 6)
Payments Analytics: (1000, 4)
Passengers Analytics: (1039, 6)


In [85]:
print("===== FINAL VALIDATION =====")
print("Flights duplicate rows:",flights_analytics.duplicated().sum())
print("Bookings duplicate rows:",bookings_analytics.duplicated().sum())
print("Payments duplicate rows:",payments_analytics.duplicated().sum())
print("Passengers duplicate rows:",passengers_analytics.duplicated().sum())
print("Flights missing values:",flights_analytics.isna().sum().sum())
print("Bookings missing values:",bookings_analytics.isna().sum().sum())
print("Payments missing values:",payments_analytics.isna().sum().sum())
print("Passengers missing values:",passengers_analytics.isna().sum().sum())

===== FINAL VALIDATION =====
Flights duplicate rows: 0
Bookings duplicate rows: 0
Payments duplicate rows: 0
Passengers duplicate rows: 0
Flights missing values: 0
Bookings missing values: 0
Payments missing values: 78
Passengers missing values: 0


In [86]:
output_folder = r"C:\Users\ajint\Downloads\ASG_Airlines_Output"
os.makedirs(output_folder, exist_ok=True)

In [87]:
flights_analytics.to_csv(os.path.join(output_folder, "flights_clean.csv"),index=False)
bookings_analytics.to_csv(os.path.join(output_folder, "bookings_clean.csv"),index=False)
payments_analytics.to_csv(os.path.join(output_folder, "payments_clean.csv"),index=False)
passengers_analytics.to_csv(os.path.join(output_folder, "passengers_clean.csv"),index=False)